In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# 1. Definir una función que ejecute UN solo notebook
def ejecutar_notebook_hijo(fila):
    tabla = fila.nombre_tabla
    ultima_fecha_carga = str(fila.ultima_fecha_carga)
    tipo_carga = fila.tipo_carga
    columna_fecha=fila.columna_fecha
    tipo_tabla = fila.tipo_tabla
    
    try:
        # Lanzamos el notebook
        resultado = dbutils.notebook.run(
            path = "/Workspace/Users/juandavid2931@gmail.com/Python-course-for-data-engineering/02_ELT/01_Ingestas/01_Metadata_Driven/03_motor_de_ingesta_tablas_v2", 
            timeout_seconds = 3600,      
            arguments = {
                "tabla": tabla,
                "ultima_fecha_carga": ultima_fecha_carga,
                "tipo_carga": tipo_carga,
                "columna_fecha":columna_fecha,
                "tipo_tabla":tipo_tabla
            }
        )
        return f" Éxito procesando {tabla}. Mensaje de salida: {resultado}"
    except Exception as e:
        return f" Error procesando {tabla}. Detalle: {e}"

# 2. Leer la tabla de metadata y traer los registros
df_metadata = spark.table("workspace.bronze.metadata")
registros = df_metadata.collect()

# 3. Configurar y lanzar la ejecución en paralelo
hilos_simultaneos = 4  # <-- Cuántas tablas quieres procesar a la vez (modifica esto según tu clúster)

print(f"🚀 Iniciando procesamiento en paralelo para {len(registros)} tablas...")

with ThreadPoolExecutor(max_workers=hilos_simultaneos) as executor:
    # Enviamos todas las filas a la función para que se ejecuten
    # Esto crea una lista de "promesas" (futuros) de ejecución
    futuros = [executor.submit(ejecutar_notebook_hijo, fila) for fila in registros]
    
    # as_completed nos permite capturar y imprimir los resultados a medida que cada notebook termina
    for futuro in as_completed(futuros):
        # future.result() nos devuelve el string de " Éxito..." o " Error..." de nuestra función
        print(futuro.result())

print("🏁 Todos los procesos han terminado.")